# Adaptive Nervous System v0.4.0

## ทดลองลดความเสียหายกรณีแย่ พร้อมเพิ่มความสมจริงของระบบ

โน้ตบุ๊กนี้รัน **Adaptive_Nervous_System_v040.zip** บน CPU ไม่ต้องใช้ GPU หรือ API key ใช้ได้ทั้งคอมพิวเตอร์และโทรศัพท์ที่เปิด Google Colab

ข้อมูลทั้งหมดเป็น **การจำลองคิวสามขั้น** ไม่มีการเชื่อมต่อระบบบริการจริงหรือส่งคำสั่งย้ายทรัพยากรภายนอก ผลคาดการณ์ของตัวควบคุมเป็นสมมติฐานของแบบจำลอง ไม่ใช่ค่าประสิทธิภาพจริงในอนาคต

การทดลองมี **704 สถานการณ์ × 6 เงื่อนไข × 7 วิธี = 29,568 episodes** ใช้ 11 ครอบครัวสถานการณ์เดิมกับ seed ใหม่ที่ตรึงไว้ มีช่วงติดตามเพิ่ม 32 นาทีทุก episode การรันเต็มและตรวจซ้ำอาจใช้เวลาหลายนาทีหรือหลายสิบนาทีตามกำลัง CPU

### วิธีที่เปรียบเทียบ

| วิธี | บทบาท |
| --- | --- |
| static | คงทรัพยากร 69 / 69 / 69 |
| capacity_v2 | ตัวเลือกจากแบบจำลอง v0.2 |
| capacity_balance | วิธีฐานที่ปรับสมดุลตามอัตราการทำงานที่สังเกต |
| v3_full | ตัวควบคุมรุ่น v0.3 ที่ตรึงไว้ |
| v4_hybrid | สมดุลการจัดสรรร่วมกับค่าปรับความเสี่ยงแบบจำกัด และการฟื้นตัวจากการค้างอยู่กับการจัดสรรเดิม |
| v4_no_risk | ตัวเทียบที่ปิดค่าปรับความเสี่ยงของ v0.4 |
| v4_no_recovery | ตัวเทียบที่ปิดกลไกฟื้นตัวของ v0.4 |

### หกเงื่อนไขที่เพิ่มข้อจำกัดทีละขั้น

1. **legacy_instant:** ทรัพยากรเป็นเศษส่วนได้ ย้ายทันที งานส่งต่อภายในนาทีเดียวกัน
2. **legacy_transit3:** เพิ่มเวลาขนย้ายทรัพยากร 3 นาที
3. **integer_transit3:** เพิ่มการจัดสรรทรัพยากรเป็นจำนวนเต็ม โดยงานและการไหลยังเป็นแบบ fluid
4. **finite_pipeline:** เพิ่มเวลาส่งต่องานหนึ่งช่วง และบัฟเฟอร์จำกัด 60,000 / 40,000 / 40,000
5. **startup:** เพิ่มช่วงเริ่มทำงานของทรัพยากรที่ย้ายเข้ามา ที่กำลัง 50% และ 75% ก่อนเต็มกำลัง
6. **delayed_noisy:** เพิ่มการสังเกตล่าช้า 2 นาที และ noise แบบ lognormal ที่มี log-space sigma 0.05

**Workflow SUCCESS ไม่เท่ากับ advancement PASS.** โน้ตบุ๊กจะประเมินแต่ละเงื่อนไขจากผลที่คุณรันจริงโดยอัตโนมัติ ไม่สรุปว่ารุ่นใหม่ผ่านเกณฑ์ล่วงหน้า และไม่รวมทั้งหกเงื่อนไขเป็นคะแนนประสิทธิภาพเดียว

รันเซลล์ตามลำดับ แพ็กเกจมี protocol ที่ตรึงแล้ว จึงใช้เฉพาะการตรวจ verify_frozen ไม่สร้างหรือแก้ freeze ใหม่ โน้ตบุ๊กตรวจ syntax และทดสอบเซลล์ตาราง/กราฟด้วยผล development ที่เก็บไว้แล้ว แต่ยังไม่ได้ทดสอบส่วนติดต่อ Colab โดยตรง


## 1. อัปโหลดแพ็กเกจ v0.4

เลือก **Adaptive_Nervous_System_v040.zip** เพียงไฟล์เดียว เซลล์ตรวจเส้นทางก่อนแตก ZIP ตรวจว่าพบโครงการเพียงชุดเดียว และเทียบขนาด/แฮชของไฟล์กับรายการแพ็กเกจ


In [ ]:
from google.colab import files
from pathlib import Path, PurePosixPath
import hashlib
import io
import json
import stat
import tempfile
import zipfile

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("กรุณาเลือก Adaptive_Nervous_System_v040.zip เพียงไฟล์เดียว")
upload_name, upload_bytes = next(iter(uploaded.items()))
if not upload_name.lower().endswith(".zip"):
    raise ValueError("ต้องเลือกไฟล์ ZIP")

EXTRACT_ROOT = Path(tempfile.mkdtemp(prefix="ans_v040_")).resolve()
ROOT_FOLDER = "Adaptive_Nervous_System_v040"
with zipfile.ZipFile(io.BytesIO(upload_bytes)) as archive:
    members = archive.infolist()
    if len(members) > 20000 or sum(m.file_size for m in members) > 1024 * 1024 * 1024:
        raise ValueError("ขนาดแพ็กเกจเกินขอบเขตที่คาดไว้")
    destinations = set()
    for member in members:
        relative = PurePosixPath(member.filename)
        target = (EXTRACT_ROOT / member.filename).resolve()
        if (not relative.parts or relative.parts[0] != ROOT_FOLDER
                or relative.is_absolute() or ".." in relative.parts
                or "\\" in member.filename
                or not target.is_relative_to(EXTRACT_ROOT)
                or stat.S_ISLNK(member.external_attr >> 16)):
            raise ValueError(f"เส้นทาง ZIP ไม่ถูกต้อง: {member.filename!r}")
        if target in destinations:
            raise ValueError(f"เส้นทางซ้ำใน ZIP: {member.filename!r}")
        destinations.add(target)
    archive.extractall(EXTRACT_ROOT)
del upload_bytes, uploaded

candidates = sorted(EXTRACT_ROOT.rglob("run_v4.py"))
if len(candidates) != 1:
    raise ValueError("ต้องพบโครงการ v0.4 เพียงหนึ่งชุด")
PROJECT = candidates[0].parent.resolve()
if PROJECT.name != ROOT_FOLDER:
    raise ValueError("ชื่อโฟลเดอร์โครงการไม่ตรงกับแพ็กเกจ v0.4")
required_files = (
    "requirements.txt", "test_lab.py", "test_capacity.py", "test_v3.py",
    "test_v4.py", "test_v4_metrics.py", "v4_protocol.json", "v4_protocol.sha256",
    "v4_selected_config.json", "freeze_v4.py", "review_v4_results.py", "evaluate_v4.py",
)
for required in required_files:
    if not (PROJECT / required).is_file():
        raise ValueError(f"แพ็กเกจไม่ครบ: {required}")

manifest_path = PROJECT / "package_manifest.json"
if not manifest_path.is_file():
    raise ValueError("ไม่พบรายการตรวจสอบแพ็กเกจ")
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
if manifest.get("version") != "0.4.0":
    raise ValueError("รายการตรวจสอบไม่ใช่แพ็กเกจ v0.4.0")
seen = set()
for entry in manifest["contents"]:
    relative = entry["path"]
    path = (PROJECT / relative).resolve()
    if relative in seen or not path.is_relative_to(PROJECT) or not path.is_file():
        raise ValueError("เส้นทางในรายการตรวจสอบไม่ถูกต้องหรือซ้ำกัน")
    seen.add(relative)
    if path.stat().st_size != entry["bytes"]:
        raise ValueError(f"ขนาดไฟล์ไม่ตรง: {relative}")
    if hashlib.sha256(path.read_bytes()).hexdigest() != entry["sha256"]:
        raise ValueError(f"Hash ไม่ตรง: {relative}")
protocol_sha = hashlib.sha256((PROJECT / "v4_protocol.json").read_bytes()).hexdigest()
if manifest.get("protocol_sha256") != protocol_sha:
    raise ValueError("Protocol ไม่ตรงกับรายการแพ็กเกจ")
print("พร้อมรัน:", PROJECT.name)
print("ตรวจไฟล์ในแพ็กเกจแล้ว:", len(seen))
print("Protocol SHA256:", protocol_sha)


## 2. ติดตั้งและทดสอบจากโค้ดชุดนี้

ติดตั้งตาม requirements แล้วตรวจ frozen protocol จากนั้นรัน **79 tests ใน 5 โมดูล**: ต้นแบบ, v0.2, v0.3, ฟิสิกส์/การสังเกต v0.4 และสถิติ v0.4

เซลล์บันทึกผลที่เกิดขึ้นจริงใน session นี้เป็น JSON และ log **นอกโฟลเดอร์โครงการ** พร้อมแฮชของโค้ดก่อน/หลัง tests ไม่คัดลอกผลทดสอบที่บรรจุมาเป็นหลักฐานใหม่ หาก test, แฮช หรือขอบเขตการทดสอบไม่ผ่าน เซลล์จะหยุดให้ตรวจรายละเอียด

การรันเซลล์นี้ซ้ำสร้างชื่อไฟล์หลักฐานใหม่ จึงไม่เขียนทับหลักฐานการรันก่อนหน้า


In [ ]:
import subprocess
import sys
import uuid

def run_python(*arguments, label=None):
    command = [sys.executable, *map(str, arguments)]
    print(label or ("กำลังรัน: " + " ".join(map(str, arguments))), flush=True)
    subprocess.run(command, cwd=PROJECT, check=True)

run_python("-m", "pip", "install", "-r", PROJECT / "requirements.txt",
           label="ติดตั้ง dependencies ของแพ็กเกจ")
run_python("-c", "from freeze_v4 import verify_frozen; print(verify_frozen()['protocol_sha256'])",
           label="ตรวจ frozen protocol โดยไม่สร้าง freeze ใหม่")

test_id = uuid.uuid4().hex[:12]
TEST_EVIDENCE = EXTRACT_ROOT / f"session_unit_tests_{test_id}.json"
TEST_LOG = EXTRACT_ROOT / f"session_unit_tests_{test_id}.log"
if TEST_EVIDENCE.resolve().is_relative_to(PROJECT):
    test_directory = Path(tempfile.mkdtemp(prefix="ans_v040_tests_"))
    TEST_EVIDENCE = test_directory / "session_unit_tests.json"
    TEST_LOG = test_directory / "session_unit_tests.log"

unit_test_script = r"""
import hashlib
import io
import json
import platform
import sys
import unittest
from pathlib import Path
import numpy as np
from freeze_v4 import FILES, verify_frozen

root = Path.cwd().resolve()
modules = ["test_lab", "test_capacity", "test_v3", "test_v4", "test_v4_metrics"]
tracked = sorted(set(FILES) | {name + ".py" for name in modules} | {"review_v4_results.py"})
def snapshot():
    return {name: hashlib.sha256((root / name).read_bytes()).hexdigest() for name in tracked}

before = snapshot()
frozen_before = verify_frozen()
log = io.StringIO()
suite = unittest.defaultTestLoader.loadTestsFromNames(modules)
result = unittest.TextTestRunner(stream=log, verbosity=2).run(suite)
after = snapshot()
unchanged = before == after
freeze_error = None
try:
    frozen_unchanged = verify_frozen() == frozen_before
except Exception as error:
    frozen_unchanged = False
    freeze_error = str(error)
passed = (result.wasSuccessful() and result.testsRun >= 79
          and len(result.skipped) == 0 and unchanged and frozen_unchanged)
evidence = {
    "passed": passed,
    "successful": result.wasSuccessful(),
    "tests_run": result.testsRun,
    "failures": len(result.failures),
    "errors": len(result.errors),
    "skipped": len(result.skipped),
    "expected_failures": len(result.expectedFailures),
    "unexpected_successes": len(result.unexpectedSuccesses),
    "modules": modules,
    "source_hashes": before,
    "sources_unchanged_during_tests": unchanged,
    "frozen_identity_unchanged": frozen_unchanged,
    "freeze_error": freeze_error,
    "protocol_sha256": frozen_before["protocol_sha256"],
    "python": platform.python_version(),
    "numpy": np.__version__,
}
log.write("\nSources unchanged during tests: " + str(unchanged) + "\n")
log.write("Frozen identity unchanged: " + str(frozen_unchanged) + "\n")
log.write("Required test scope passed: " + str(passed) + "\n")
Path(sys.argv[1]).write_text(json.dumps(evidence, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
Path(sys.argv[2]).write_text(log.getvalue(), encoding="utf-8")
sys.stdout.write(log.getvalue())
raise SystemExit(0 if passed else 1)
"""
run_python("-c", unit_test_script, TEST_EVIDENCE, TEST_LOG,
           label="รัน tests ใหม่และบันทึกหลักฐานของ session นี้")
fresh_tests = json.loads(TEST_EVIDENCE.read_text(encoding="utf-8"))
print("Tests ที่รันจริง:", fresh_tests["tests_run"], "| Passed:", fresh_tests["passed"])
print("หลักฐาน:", TEST_EVIDENCE.name, "และ", TEST_LOG.name)


## 3. รัน validation และประเมินเกณฑ์โดยอัตโนมัติ

ทุกครั้งที่รันเซลล์นี้ใช้โฟลเดอร์ผลใหม่ การรันซ้ำเป็นการตรวจความทำซ้ำได้ ไม่เพิ่มจำนวนสถานการณ์อิสระทางสถิติ

ช่วงประเมินเดิมยาว 60 หรือ 96 นาที ตามด้วย **32 นาทีที่ไม่มีงานเข้าใหม่และคง productivity จริงแถวสุดท้าย** เงื่อนไขนี้เป็นสมมติฐานสำหรับวัดการระบายงาน ไม่ได้บอกเวลาจบหรืออนาคตให้ตัวควบคุม

เมื่อ workflow เสร็จ จะตรวจแถวผลทั้งหมด, replay ตัวแทน **462 traces / 51,576 นาที**, ตรวจการส่งข้อมูลและสถิติด้วย reviewer แล้วเรียก evaluator ตามกติกาที่ตรึงไว้ พร้อมผล tests ใหม่ของคุณ

**เกณฑ์หลัก:** ใน delayed_noisy ต้องลด CVaR90 ของ harm เทียบ Static ได้ดีกว่า v3_full โดยขอบล่างของช่วง 95% มากกว่าศูนย์ เกณฑ์ advancement ยังตรวจความเสี่ยงทุกเงื่อนไข, mean noninferiority เทียบ V2/Balance ภายใน allowance 5%, งานสูญหาย/งานค้างปลายช่วง/ช่วงติดตาม, underload และความครบถ้วนของหลักฐาน ไม่ปรับเกณฑ์ตามผลที่พบ


In [ ]:
from datetime import datetime, timezone

if not TEST_EVIDENCE.is_file() or not TEST_LOG.is_file():
    raise RuntimeError("กรุณารันเซลล์ tests เพื่อสร้างหลักฐานใหม่ของ session นี้")
fresh_tests = json.loads(TEST_EVIDENCE.read_text(encoding="utf-8"))
if fresh_tests.get("passed") is not True or fresh_tests.get("protocol_sha256") != protocol_sha:
    raise RuntimeError("หลักฐาน tests ยังไม่ผ่านหรือไม่ตรงกับ protocol ชุดนี้")

run_name = (datetime.now(timezone.utc).strftime("validation_%Y%m%d_%H%M%S")
            + "_" + uuid.uuid4().hex[:8])
OUTPUT_ROOT = Path("/content/ans_v040_runs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RESULTS = OUTPUT_ROOT / run_name
if RESULTS.exists():
    raise FileExistsError("ต้องใช้โฟลเดอร์ใหม่สำหรับการทดลอง")

run_python("run_v4.py", "--split", "validation", "--out", RESULTS,
           label="รัน 29,568 episodes บน CPU; กรุณารอให้ workflow เสร็จ")
for filename in ("summary.json", "episodes.csv", "verification.json"):
    if not (RESULTS / filename).is_file():
        raise RuntimeError(f"ไม่พบผลลัพธ์: {filename}")

run_python("review_v4_results.py", "--out", RESULTS,
           "--tests", TEST_EVIDENCE, "--test-log", TEST_LOG,
           "--report", RESULTS / "independent_review.json",
           label="ตรวจผลและ replay อย่างอิสระโดยใช้ tests ใหม่ของ session นี้")
run_python("evaluate_v4.py", "--results", RESULTS, "--split", "validation",
           "--protocol", PROJECT / "v4_protocol.json",
           "--review", RESULTS / "independent_review.json",
           "--tests", TEST_EVIDENCE, "--out", RESULTS / "decision.json",
           label="ประเมินเกณฑ์ acceptance ที่ตรึงไว้โดยอัตโนมัติ")

verification = json.loads((RESULTS / "verification.json").read_text(encoding="utf-8"))
decision = json.loads((RESULTS / "decision.json").read_text(encoding="utf-8"))
print("\nWorkflow execution:", verification["status"])
print("Primary risk criterion:", "PASS" if decision["primary"]["passed"] else "FAIL")
print("Advancement:", "PASS" if decision["advancement_passed"] else "FAIL")
if decision["failed_criteria"]:
    print("เงื่อนไขที่ไม่ผ่าน:", ", ".join(decision["failed_criteria"]))
print("Workflow สำเร็จไม่ได้ยืนยันว่า advancement ผ่าน")
print("ผลการรันนี้:", run_name)


## 4. อ่าน deficit ร่วมกับงานที่ยังอยู่และงานที่สูญหาย

**Demand deficit = งานที่เสนอเข้าระบบสะสม − งานที่เสร็จสะสม** และเท่ากับ **งานที่ยังอยู่ในคิว/กำลังส่งต่อ + งานที่ถูกทิ้งสะสม** การทิ้งงานจึงไม่ทำให้ตัวชี้วัดหลักดูดีขึ้นเอง

Mean deficit และ mean retained นับช่วงประเมินเดิม ส่วนต่างคือค่าเฉลี่ยของงานที่ถูกทิ้งสะสมในแต่ละนาที ไม่ใช่จำนวนงานที่ถูกทิ้งทั้งหมด ณ ปลายช่วง ตารางยังแสดง completion และ dropped work แยกไว้ การลด deficit จึงไม่ควรถูกเรียกเป็นเปอร์เซ็นต์เพิ่ม throughput โดยอัตโนมัติ

ทุกตารางและกราฟด้านล่างแยกตามเงื่อนไข ไม่เฉลี่ยรวมทั้งหกเงื่อนไข


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

summary = json.loads((RESULTS / "summary.json").read_text(encoding="utf-8"))
if summary.get("version") != "0.4.0":
    raise ValueError("ผลลัพธ์ไม่ใช่ v0.4")
policies = [
    "static", "capacity_v2", "capacity_balance", "v3_full",
    "v4_hybrid", "v4_no_risk", "v4_no_recovery",
]
names = {
    "static": "Static", "capacity_v2": "Capacity v0.2",
    "capacity_balance": "Observed-rate balance", "v3_full": "v0.3 full",
    "v4_hybrid": "v0.4 hybrid", "v4_no_risk": "v0.4 no risk penalty",
    "v4_no_recovery": "v0.4 no recovery",
}
regime_names = {
    "legacy_instant": "1. ย้ายทันที / fluid",
    "legacy_transit3": "2. ขนย้ายทรัพยากร 3 นาที",
    "integer_transit3": "3. เพิ่มทรัพยากรจำนวนเต็ม",
    "finite_pipeline": "4. เพิ่ม pipeline และบัฟเฟอร์จำกัด",
    "startup": "5. เพิ่มช่วงเริ่มทำงาน",
    "delayed_noisy": "6. เพิ่มข้อมูลล่าช้าและ noise",
}
if set(summary["regimes"]) != set(regime_names):
    raise ValueError("ผลไม่ครบหกเงื่อนไข")
print("Scenarios:", summary["scenarios"], "| Episodes:", summary["episodes"])
print("Original minutes:", summary["original_minutes"],
      "| Follow-up minutes:", summary["followup_minutes"])

FIGURES = RESULTS / "notebook_figures"
FIGURES.mkdir(exist_ok=True)
for regime, label in regime_names.items():
    display(Markdown("### " + label))
    overall = pd.DataFrame.from_dict(summary["regimes"][regime]["overall"], orient="index").reindex(policies)
    table = overall[["mean_deficit", "mean_queue", "end_deficit", "completed", "dropped"]].copy()
    table.insert(2, "mean_accumulated_drops", overall["mean_deficit"] - overall["mean_queue"])
    display(table.rename(index=names, columns={
        "mean_deficit": "Mean deficit H", "mean_queue": "Mean retained H",
        "mean_accumulated_drops": "Mean accumulated drops H",
        "end_deficit": "Deficit at H", "completed": "Completed H", "dropped": "Dropped H",
    }).round(3))

    retained = overall["mean_queue"].to_numpy()
    lost = (overall["mean_deficit"] - overall["mean_queue"]).clip(lower=0).to_numpy()
    fig, ax = plt.subplots(figsize=(9, 4.6), constrained_layout=True)
    labels = [names[policy] for policy in policies]
    ax.barh(labels, retained, color="#366D95", label="Retained work (queues + pipeline)")
    ax.barh(labels, lost, left=retained, color="#C58236", label="Accumulated dropped work")
    ax.invert_yaxis()
    ax.set_title(f"{regime}: original-horizon mean demand deficit", pad=38)
    ax.set_xlabel("Work units; lower is better")
    ax.grid(axis="x", alpha=.2)
    ax.set_axisbelow(True)
    ax.legend(loc="lower center", bbox_to_anchor=(.5, 1.01), ncol=2, frameon=False, fontsize=8)
    fig.savefig(FIGURES / f"{regime}_mean_deficit.png", dpi=150)
    plt.show()


## 5. ประโยชน์เฉลี่ย ความเสี่ยง และเกณฑ์ PASS/FAIL

ตารางเปรียบเทียบแสดง v4_hybrid เทียบ V3, V2, Balance, Static และตัวเทียบที่ปิดกลไก ค่า reduction เป็นบวกหมายถึง mean deficit ต่ำกว่า baseline ต้องอ่านข้อแลกเปลี่ยนกับ V2/Balance แม้ผลเทียบ V3 จะดีขึ้น

**Harm ของแต่ละวิธี = max(mean deficit ของวิธีนั้น − mean deficit ของ Static, 0).** CVaR90 คือค่าเฉลี่ย harm ของกรณีแย่ที่สุด 10% โดยรวมส่วนขอบแบบเศษส่วนเมื่อจำเป็น ไม่ใช่ความน่าจะเป็นที่ระบบปลอดภัย และไม่ใช่ harm ที่คำนวณโดยลบ V3 ออกจาก V4 โดยตรง

ช่วง 95% มาจาก paired bootstrap 2,000 ครั้ง โดยสุ่มภายในแต่ละครอบครัวและเก็บคู่ทุกวิธีไว้ ค่า relative effect ที่หารด้วย baseline ศูนย์จะแสดงว่าง ให้ใช้ความต่างแบบ absolute แทน ช่วงของการเปรียบเทียบรองเป็นข้อมูลประกอบที่ไม่ได้ปรับสำหรับหลายการเปรียบเทียบ


In [ ]:
comparison_order = [
    "v3_full", "capacity_v2", "capacity_balance", "static", "v4_no_risk", "v4_no_recovery",
]
for regime, label in regime_names.items():
    part = summary["regimes"][regime]
    display(Markdown("### " + label))
    comparisons = []
    for baseline in comparison_order:
        item = part["comparisons"]["v4_hybrid"][baseline]
        ci = item["reduction_pct_ci95"]
        comparisons.append({
            "Baseline": names[baseline],
            "Deficit reduction (%)": item["reduction_pct"],
            "95% low (%)": ci[0] if ci else None,
            "95% high (%)": ci[1] if ci else None,
            "Absolute mean improvement": item["mean_improvement"],
            "Wins / ties / losses": f"{item['wins']} / {item['ties']} / {item['losses']}",
        })
    display(pd.DataFrame(comparisons).round(3))
    risk = pd.DataFrame.from_dict(part["static_relative_harm"], orient="index").reindex(policies)
    risk["frequency"] *= 100
    display(risk[["frequency", "mean_positive_harm", "cvar90", "maximum_harm"]].rename(
        index=names, columns={
            "frequency": "Worse than Static (%)",
            "mean_positive_harm": "Mean positive harm",
            "cvar90": "CVaR90 vs Static",
            "maximum_harm": "Maximum harm",
        }).round(3))
    risk_comparison = part["static_harm_comparisons"]["v4_hybrid"]["v3_full"]["cvar90"]
    print("CVaR reduction: V3 static-relative risk minus V4 static-relative risk =",
          round(risk_comparison["risk_reduction"], 3),
          "| 95% CI:", [round(x, 3) for x in risk_comparison["risk_reduction_ci95"]])

decision = json.loads((RESULTS / "decision.json").read_text(encoding="utf-8"))
display(Markdown("### ผลตามเกณฑ์ที่ตรึงไว้"))
print("Workflow:", verification["status"],
      "| Advancement:", "PASS" if decision["advancement_passed"] else "FAIL")
decision_rows = [
    {"Criterion": name, "Result": "PASS" if value["passed"] else "FAIL"}
    for name, value in decision["advancement_checks"].items()
]
display(pd.DataFrame(decision_rows))
for section, title in (
    ("risk_point_nonworsening", "Risk point checks"),
    ("mean_noninferiority", "Mean noninferiority checks: fixed 5% allowance"),
    ("loss_and_endpoint_point_nonworsening", "Dropped work / terminal / follow-up checks"),
):
    checks = decision[section]["checks"]
    failed = [item for item in checks if not item["passed"]]
    print(title, "—", len(checks) - len(failed), "/", len(checks), "ผ่าน")
    if failed:
        display(pd.DataFrame(failed).round(3))
if decision["integrity"]["missing_or_failed"]:
    print("หลักฐาน integrity ที่ขาดหรือไม่ผ่าน:", decision["integrity"]["missing_or_failed"])
if decision["underload"]["violations"]:
    display(pd.DataFrame(decision["underload"]["violations"]))
print("Allowance 5% ใช้กับ mean deficit เท่านั้น ไม่ได้ให้ margin เพิ่มแก่ CVaR หรือ dropped work")
print("การผ่านเกณฑ์ในแบบจำลองนี้ยังไม่ใช่ข้อยืนยันสำหรับนำไปควบคุมระบบจริง")


## 6. ช่วงติดตาม การระบายงาน และต้นทุนทางกายภาพ

**Drain time** วัดเวลาที่งานซึ่งยังอยู่ในคิวและ pipeline หมด หลังจบช่วงประเมินเดิม โดยจำกัดการสังเกตไว้ 32 นาที ต้องอ่านคู่กับ **censored / uncleared fraction** เพราะบางกรณียังไม่ทราบเวลาระบายจริง งานที่ถูกทิ้งแล้วไม่ถือว่าเสร็จ และ deficit อาจยังเป็นบวกแม้งานที่ยังอยู่ระบายหมด

**Transit resource-minutes** นับทรัพยากรที่ยังขนย้ายและทำงานไม่ได้ ส่วน **startup capacity-minutes** นับ active − effective ในช่วงเริ่มทำงาน แสดงแยกกันเพื่อไม่คิดต้นทุนซ้ำ ไม่ใช่ค่าใช้จ่ายเป็นเงิน

ตารางครอบครัวท้ายแต่ละเงื่อนไขช่วยดูว่าผลเฉลี่ยกระจุกตัวอยู่ที่ปัญหาบางแบบหรือไม่ แสดงครบทั้ง 11 ครอบครัว รวมกรณีที่รุ่นใหม่แย่ลง


In [ ]:
for regime, label in regime_names.items():
    part = summary["regimes"][regime]
    overall = pd.DataFrame.from_dict(part["overall"], orient="index").reindex(policies)
    display(Markdown("### " + label))
    followup = overall[[
        "tail_auc_deficit", "tail_end_deficit", "tail_end_queue",
        "drain_time_capped", "drain_uncleared_fraction", "total_dropped",
    ]].copy()
    followup["drain_uncleared_fraction"] *= 100
    display(followup.rename(index=names, columns={
        "tail_auc_deficit": "Tail deficit AUC",
        "tail_end_deficit": "Final deficit",
        "tail_end_queue": "Final retained work",
        "drain_time_capped": "Drain time capped 32 min",
        "drain_uncleared_fraction": "Censored (%)",
        "total_dropped": "Total dropped H+32",
    }).round(3))
    exposure = overall[[
        "moves", "total_moves",
        "resource_minutes_lost", "startup_capacity_minutes_lost",
        "tail_resource_minutes_lost", "tail_startup_capacity_minutes_lost",
    ]]
    display(exposure.rename(index=names, columns={
        "moves": "Moves H", "total_moves": "Moves H+32",
        "resource_minutes_lost": "Transit loss H",
        "startup_capacity_minutes_lost": "Startup loss H",
        "tail_resource_minutes_lost": "Transit loss tail",
        "tail_startup_capacity_minutes_lost": "Startup loss tail",
    }).round(3))
    families = []
    for group, group_part in part["by_group"].items():
        hybrid = group_part["overall"]["v4_hybrid"]
        row = {
            "Family": group,
            "Hybrid mean deficit": hybrid["mean_deficit"],
            "Hybrid total drops": hybrid["total_dropped"],
        }
        for baseline in ("v3_full", "capacity_v2", "capacity_balance"):
            row[f"Reduction vs {names[baseline]} (%)"] = (
                group_part["comparisons"]["v4_hybrid"][baseline]["reduction_pct"]
            )
        families.append(row)
    display(pd.DataFrame(families).round(3))


## 7. ทางเลือก: ตรวจทำซ้ำเต็มอีกครั้ง

ขั้นตอนนี้ **ปิดไว้เป็นค่าเริ่มต้น** และไม่จำเป็นต่อการอ่านผลรอบแรก หากต้องการตรวจทำซ้ำ ให้เปลี่ยน RUN_REPRODUCIBILITY เป็น True แล้วรันเซลล์ถัดไป จะใช้เวลาเพิ่มใกล้เคียงกับ validation เต็มอีกหนึ่งรอบ

การตรวจเปรียบเทียบไฟล์ผล deterministic 88 ไฟล์ระหว่างโฟลเดอร์ใหม่สองชุด โดยไม่คาดหวังว่าเวลาเหตุการณ์หรือระยะเวลารันจะตรงกัน ผลรอบซ้ำเป็น **หลักฐานการทำซ้ำ ไม่ใช่สถานการณ์อิสระเพิ่มอีก 704 กรณี** และไม่เปลี่ยนผล acceptance ของรอบแรก

ZIP รอบซ้ำดาวน์โหลดแยก ส่วนรายงานเปรียบเทียบจะรวมอยู่ใน ZIP ผลหลักด้านล่าง


In [ ]:
RUN_REPRODUCIBILITY = False

if RUN_REPRODUCIBILITY:
    import shutil
    repeat_name = run_name + "_repeat_" + uuid.uuid4().hex[:8]
    REPEAT_RESULTS = OUTPUT_ROOT / repeat_name
    if REPEAT_RESULTS.exists():
        raise FileExistsError("ต้องใช้โฟลเดอร์ใหม่สำหรับการรันซ้ำ")
    if not (PROJECT / "check_v4_reproducibility.py").is_file():
        raise RuntimeError("ไม่พบตัวตรวจทำซ้ำในแพ็กเกจ")
    run_python("run_v4.py", "--split", "validation", "--out", REPEAT_RESULTS,
               label="รันซ้ำแบบเลือกเพิ่มเติม: ไม่เพิ่มจำนวนสถานการณ์อิสระ")
    reproduction_report = RESULTS / ("reproducibility_" + uuid.uuid4().hex[:8] + ".json")
    run_python("check_v4_reproducibility.py", "--first", RESULTS,
               "--second", REPEAT_RESULTS, "--report", reproduction_report,
               label="เปรียบเทียบไฟล์ deterministic 88 ไฟล์จากสอง workflow")
    reproduction = json.loads(reproduction_report.read_text(encoding="utf-8"))
    print("Reproducibility:", reproduction["status"],
          "| Files:", reproduction["files_compared"],
          "| Byte-identical:", reproduction["all_byte_identical"])
    shutil.copy2(TEST_EVIDENCE, REPEAT_RESULTS / "session_unit_tests.json")
    shutil.copy2(TEST_LOG, REPEAT_RESULTS / "session_unit_tests.log")
    shutil.copy2(reproduction_report, REPEAT_RESULTS / "reproducibility.json")
    repeat_archive = Path(shutil.make_archive(
        str(OUTPUT_ROOT / (repeat_name + "_results")), "zip", root_dir=REPEAT_RESULTS
    ))
    files.download(str(repeat_archive))
else:
    print("ข้ามการรันซ้ำเพิ่มเติม; ผล validation และ acceptance รอบแรกยังอยู่ครบ")


## 8. ดาวน์โหลดผลที่รันใหม่

ZIP ผลลัพธ์รวม episode rows/CSV, summary, decision, independent review, traces ตัวแทน, configuration, หลักฐาน workflow, **ผล tests และ log ใหม่ของ session นี้** รวมทั้งกราฟที่แสดงข้างบน

เก็บ ZIP ผลลัพธ์คู่กับ **Adaptive_Nervous_System_v040.zip** ต้นฉบับ โค้ดและ protocol ที่ตรึงอยู่ในแพ็กเกจต้นฉบับ การรันซ้ำบนสถานการณ์เดิมเป็นการตรวจทำซ้ำ ไม่เพิ่มขนาดชุดหลักฐานอิสระ


In [ ]:
import shutil

for filename in ("summary.json", "decision.json", "independent_review.json", "verification.json"):
    if not (RESULTS / filename).is_file():
        raise RuntimeError(f"ผลยังไม่ครบ กรุณารันขั้นตอนก่อนหน้า: {filename}")
shutil.copy2(TEST_EVIDENCE, RESULTS / "session_unit_tests.json")
shutil.copy2(TEST_LOG, RESULTS / "session_unit_tests.log")
results_archive = Path(shutil.make_archive(
    str(OUTPUT_ROOT / (run_name + "_results")), "zip", root_dir=RESULTS
))
print("ดาวน์โหลด:", results_archive.name)
print("Workflow:", verification["status"],
      "| Advancement:", "PASS" if decision["advancement_passed"] else "FAIL")
files.download(str(results_archive))
